# 📈 Airflow DAG: `stock_model_trainer_mlflow`
Trainiert und verwaltet ML-Modelle zur Vorhersage von Aktienrenditen auf Basis historischer Kursdaten. Die Modelle werden mit LightGBM trainiert und zentral in **MLflow** geloggt und registriert.

## 🚀 Überblick
Dieser DAG automatisiert die folgenden Schritte:
1. **Kombinieren von Rohdaten**: Aggregiert alle Einzel-Aktien-CSV-Dateien zu einer gemeinsamen Datei (`combined_stock_data.csv`).
2. **Feature Engineering**: Extrahiert zeitbasierte Features (z. B. Stunde, Wochentag, Monat) und berechnet Renditen (`return`).
3. **Modelltraining**: Für jede Aktie wird ein separates LightGBM-Modell trainiert.
4. **MLflow-Logging**:
   - Logging von Parametern, Metriken und Modell-Artefakten
   - Registrierung im Model Registry unter `stock_price_model`
   - Automatische Promotion in den `Production`-Stage
   - Logging einer begleitenden `info.json` mit Feature-Definition

## ⚙️ Setup
### Verzeichnisse
| Pfad | Beschreibung |
|------|--------------|
| `/home/holu/airflow/data/stock_data/` | Einzelne Aktien-CSV-Dateien & kombinierte Datei |
| `/home/holu/airflow/data/models/`     | MLflow-Artefakte & `info.json` |

### MLflow Tracking Server
Stelle sicher, dass MLflow lokal unter folgendem URI läuft:

`http://localhost:5001`

```bash
mlflow server \
  --backend-store-uri ./mlruns \
  --default-artifact-root ./artifacts \
  --host 0.0.0.0 \
  --port 5001
```

## 📅 Zeitplan
| Task | Cron Expression | Bedeutung |
|------|------------------|-----------|
| `combine_stock_csvs_task` | `10 * * * *` | Startet stündlich um xx:10 |
| `train_and_log_model_task` | direkt danach | Modelltraining & Logging |

## 🧪 Feature-Set (aktuell)
Die Modelle nutzen folgende Features:
- `Open`, `High`, `Low`, `Volume`
- `hour`, `dayofweek`, `month`

Das Ziel (`target`) ist die prozentuale Rendite der Aktie zur Vorperiode:
```python
return = (Close_t - Close_t-1) / Close_t-1
```

## 📦 Output in MLflow
Für jedes Ticker-Modell wird geloggt:
- Modell: LightGBM-Regressor
- Metriken: `r2`, `rmse`
- Artefakte: `model`, `info/info.json`
- Stage: automatisch → `Production`

## ✅ Voraussetzungen
- Airflow (>=2.6)
- MLflow (>=2.x)
- LightGBM
- pandas, scikit-learn, numpy

Optional: Nutzung des DAGs im Zusammenspiel mit einem FastAPI-Modellserver (`serve_model_api`), um die Vorhersagen via REST bereitzustellen.

## 📂 Beispielstruktur
```bash
data/
├── stock_data/
│   ├── AAPL.csv
│   ├── MSFT.csv
│   ├── combined_stock_data.csv  # Auto-generiert
├── models/
│   └── info.json                # Feature-Dokumentation
```

## 🧠 Hinweise
- Die DAG setzt voraus, dass CSV-Dateien mit Spalten wie `Datetime`, `Open`, `Close`, etc. vorhanden sind.
- Wenn `r2 < 0.0`, wird das Modell trotzdem geloggt (zur Diagnose), aber mit Warnung.
- `info.json` ist erforderlich für den Produktionsserver, da dort Feature-Spalten dynamisch eingelesen werden.

## 🧩 Erweiterungsmöglichkeiten
- ❄️ Great Expectations für Datenvalidierung
- 📉 Automatisierte Retrain-Triggers bei Modell-Drift
- ☁️ MLflow Remote-Tracking (z. B. S3, PostgreSQL)

## ✍️ Autor
Elvir Ibrahimovic  
Letzte Änderung: `17.05.2025, 00:01`